# CrossCat: several clusterings of the same animals

Figure 21.22 of the book shows the MAP estimate of CrossCat (Shafto et al., 2006, *Learning cross-cutting systems
of categories*) on a matrix of animals $\times$ binary features. A single clustering, as in biclustering
([biclustering_irm.ipynb](biclustering_irm.ipynb), figure 21.20), assumes that one grouping of the animals explains
every feature. CrossCat instead splits the features into **views**, and clusters the animals separately in each
view: in the figure, one view groups the animals taxonomically (mammals, reptiles, birds, insects), another by
ecology (predators, flyers, ...), and a third holds idiosyncratic features that hardly cluster at all.

The figure's data (22 animals $\times$ 106 features) is not publicly available, so this notebook applies the same
model to the animal data of figure 21.20: 50 animals $\times$ 85 binary features (Osherson et al., 1991, extended by
Kemp et al., 2006), distributed with the *Animals with Attributes 2* dataset.

In [ ]:
import hashlib
import io
import os
import urllib.request
import zipfile

import numpy as np
from matplotlib import pyplot as plt
from scipy.special import betaln, gammaln

try:
    import probml_utils as pml
except ModuleNotFoundError:
    %pip install -qq git+https://github.com/probml/probml-utils.git
    import probml_utils as pml

## Data

The same download as [biclustering_irm.ipynb](biclustering_irm.ipynb) (32 kB, checked by its SHA-256).

In [ ]:
url = "https://cvml.ist.ac.at/AwA2/AwA2-base.zip"
sha256 = "3f02f8a446e81eff4f9f7d2ebb3e10b73f7c4bba3b5fd7b3442a37d38f013bcc"
data_dir = "data/awa2"
zip_path = os.path.join(data_dir, "AwA2-base.zip")
os.makedirs(data_dir, exist_ok=True)
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
with open(zip_path, "rb") as fp:
    assert hashlib.sha256(fp.read()).hexdigest() == sha256, f"unexpected file: {zip_path}"

with zipfile.ZipFile(zip_path) as zf:  # read the three text files without extracting the archive
    read = lambda name: zf.read(f"Animals_with_Attributes2/{name}").decode()
    X = np.loadtxt(io.StringIO(read("predicate-matrix-binary.txt")), dtype=int)
    animals = [line.split()[1].replace("+", " ") for line in read("classes.txt").splitlines()]
    features = [line.split()[1] for line in read("predicates.txt").splitlines()]
n, d = X.shape
print(f"{n} animals x {d} features")

## The model

- The features are partitioned into views: $\mathbf{v} \sim \mathrm{CRP}(\alpha_v)$.
- In each view $k$, the animals are partitioned: $\mathbf{z}^{(k)} \sim \mathrm{CRP}(\alpha)$, independently across views.
- Each feature $j$ in view $k$ has its own probability in each animal cluster $c$:
  $\theta_{jc} \sim \mathrm{Beta}(\beta, \beta)$, $x_{ij} \sim \mathrm{Ber}(\theta_{j, z^{(k)}_i})$.

Within a view this is a Dirichlet process mixture of independent Bernoullis; with one view it is that mixture on the
whole matrix. Integrating out $\theta$, a feature with $n^1_{c}$ ones and $n^0_{c}$ zeros among the animals of cluster
$c$ contributes $\prod_c B(\beta + n^1_c, \beta + n^0_c) / B(\beta, \beta)$.

**Gibbs sampling** alternates two moves:

- **Animals within a view:** remove animal $i$ from its cluster in view $k$ and reassign it, with probability
  proportional to the cluster size times the predictive probability of its features in that view (or $\alpha$
  times that of a new cluster).
- **Features between views:** remove feature $j$ from its view and reassign it to view $k$ with probability
  proportional to the number of features in $k$ times the marginal likelihood of feature $j$ under $k$'s clustering
  of the animals. A new view needs a clustering of its own: one is drawn from the CRP prior and scored with weight
  $\alpha_v$ (Neal's algorithm 8 with one auxiliary view).

We fix $\alpha = \alpha_v = 1$ and $\beta = 0.5$, run 3 chains of 200 sweeps from one view, and keep the state
with the highest joint probability (the MAP estimate the book's figure shows).

In [ ]:
def relabel(z):
    return np.unique(z, return_inverse=True)[1]


def crp_logp(z, alpha):
    m = np.bincount(z)
    return len(m) * np.log(alpha) + gammaln(m).sum() + gammaln(alpha) - gammaln(alpha + len(z))


def crp_sample(n, alpha, rng):
    z = np.zeros(n, dtype=int)
    for i in range(1, n):
        p = np.append(np.bincount(z[:i]), alpha)
        z[i] = rng.choice(len(p), p=p / p.sum())
    return z


def feature_loglik(Xv, z, beta):
    """log p(x_j | z) for every column j of Xv, theta integrated out."""
    Zr = np.eye(z.max() + 1)[z]
    N1 = Zr.T @ Xv
    N0 = Zr.sum(0)[:, None] - N1
    return np.sum(betaln(beta + N1, beta + N0) - betaln(beta, beta), axis=0)


def log_joint(X, v, Z, alpha, alpha_v, beta):
    lp = crp_logp(v, alpha_v)
    for k, z in enumerate(Z):
        lp += crp_logp(z, alpha) + feature_loglik(X[:, v == k], z, beta).sum()
    return lp


def sweep_animals(Xv, z, alpha, beta, rng):
    for i in rng.permutation(len(z)):
        others = np.arange(len(z)) != i
        z_o = relabel(z[others])
        Zr = np.eye(z_o.max() + 1)[z_o]
        N1, m = Zr.T @ Xv[others], Zr.sum(0)
        same = np.where(Xv[i] == 1, N1, m[:, None] - N1)  # animals in each cluster with the same value
        logp = np.log(m) + np.sum(np.log(beta + same) - np.log(2 * beta + m[:, None]), axis=1)
        logp = np.append(logp, np.log(alpha) + Xv.shape[1] * np.log(0.5))  # a new cluster
        p = np.exp(logp - logp.max())
        k = rng.choice(len(p), p=p / p.sum())
        z[others], z[i] = z_o, k
    return relabel(z)


def sweep_features(X, v, Z, alpha, alpha_v, beta, rng):
    for j in rng.permutation(X.shape[1]):
        x = X[:, [j]]
        counts = np.bincount(np.delete(v, j), minlength=len(Z))
        logp = [np.log(c) + feature_loglik(x, z, beta)[0] if c > 0 else -np.inf for c, z in zip(counts, Z)]
        z_new = relabel(crp_sample(len(x), alpha, rng))  # auxiliary view
        logp = np.append(logp, np.log(alpha_v) + feature_loglik(x, z_new, beta)[0])
        p = np.exp(logp - logp.max())
        k = rng.choice(len(p), p=p / p.sum())
        if k == len(Z):
            Z.append(z_new)
        v[j] = k
        used = np.unique(v)  # drop a view left empty
        Z[:] = [Z[k] for k in used]
        v[:] = np.searchsorted(used, v)
    return v, Z


def crosscat(X, alpha=1.0, alpha_v=1.0, beta=0.5, n_sweeps=200, seed=0):
    rng = np.random.default_rng(seed)
    v = np.zeros(X.shape[1], dtype=int)  # one view
    Z = [relabel(rng.integers(5, size=X.shape[0]))]
    trace, best = [], (-np.inf, None, None)
    for _ in range(n_sweeps):
        for k in range(len(Z)):
            Z[k] = sweep_animals(X[:, v == k], Z[k], alpha, beta, rng)
        v, Z = sweep_features(X, v, Z, alpha, alpha_v, beta, rng)
        trace.append(log_joint(X, v, Z, alpha, alpha_v, beta))
        if trace[-1] > best[0]:
            best = (trace[-1], v.copy(), [z.copy() for z in Z])
    return best, np.array(trace)


runs = [crosscat(X, seed=seed) for seed in range(3)]
traces = [trace for _, trace in runs]
lp, v, Z = max((best for best, _ in runs), key=lambda b: b[0])
for seed, ((lp_s, v_s, Z_s), _) in enumerate(runs):
    print(f"chain {seed}: best log joint {lp_s:.1f}, {len(Z_s)} views with "
          f"{', '.join(str(s) for s in sorted(np.bincount(v_s), reverse=True))} features")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for seed, (trace, color) in enumerate(zip(traces, ["#2a78d6", "#eb6834", "#1baf7a"])):
    ax.plot(np.arange(1, len(trace) + 1), trace, color=color, lw=1.5, label=f"chain {seed}")
ax.set_xlabel("Gibbs sweep")
ax.set_ylabel("log joint probability")
ax.legend()
fig.tight_layout()
pml.savefig("crosscat_animals_trace.pdf")
plt.show()

## The views

For each view (largest first): its features, and its clusters of animals.

In [ ]:
view_order = np.argsort(-np.bincount(v), kind="stable")
for n_view, k in enumerate(view_order, 1):
    z = Z[k]
    print(f"View {n_view} ({np.sum(v == k)} features): " + ", ".join(features[j] for j in np.where(v == k)[0]))
    for c in np.argsort(-np.bincount(z), kind="stable"):
        print("    " + ", ".join(animals[i] for i in np.where(z == c)[0]))
    print()

## The figure

As in figure 21.22: one panel per view, the animals sorted by that view's clusters (black lines between clusters),
each with its own order. Grey = 1.

In [ ]:
n_show = min(3, len(view_order))
widths = [np.sum(v == k) for k in view_order[:n_show]]
fig, axs = plt.subplots(1, n_show, figsize=(16, 10), gridspec_kw={"width_ratios": widths}, squeeze=False)
for ax, k, letter in zip(axs[0], view_order[:n_show], "ABC"):
    z = Z[k]
    cols = np.where(v == k)[0]
    cluster_order = np.argsort(-np.bincount(z), kind="stable")
    rows = np.concatenate([np.where(z == c)[0] for c in cluster_order])
    ax.imshow(X[np.ix_(rows, cols)], cmap="Greys", vmin=0, vmax=1.6, aspect="auto", interpolation="nearest")
    for b in np.cumsum(np.bincount(z)[cluster_order])[:-1]:
        ax.axhline(b - 0.5, color="black", lw=3)
    ax.set_yticks(np.arange(n))
    ax.set_yticklabels([animals[i] for i in rows], fontsize=7)
    ax.set_xticks(np.arange(len(cols)))
    ax.set_xticklabels([features[j] for j in cols], rotation=90, fontsize=7)
    ax.set_title(letter, loc="left", fontsize=16, fontweight="bold")
fig.tight_layout()
pml.savefig("crosscat_animals.pdf")
plt.show()

## One view against several

The best state with all features forced into one view (a Dirichlet process mixture of the animals) scores lower:
no single grouping of the animals explains every feature as well as several groupings do.

In [ ]:
rng = np.random.default_rng(0)
z1, best_one = relabel(rng.integers(5, size=n)), -np.inf
for _ in range(200):
    z1 = sweep_animals(X, z1, 1.0, 0.5, rng)
    best_one = max(best_one, log_joint(X, np.zeros(d, dtype=int), [z1], 1.0, 1.0, 0.5))
print(f"best log joint: one view {best_one:.1f}, CrossCat {lp:.1f} ({len(Z)} views)")

As in figure 21.22, the largest view groups the animals much like a taxonomy (whales and seals, hoofed grazers,
predators, rodents, primates), while the smaller views group the same animals differently, by the features those
views hold, and some views separate only one small group from everything else, as the frog is separated in panel B.

The sampler fixes $\alpha$, $\alpha_v$ and $\beta$, which Shafto et al. sample too, and moves one animal or feature
at a time, so different chains can end in different numbers of views; the notebook keeps the best of three.